# `pt.pivot` / `df.pt.pivot` — Excel-Style 2D Pivot Tables

`pt.pivot()` summarizes and aggregates tabular data across dimensions using the intuitive **RCVA** grammar:
- **`r`** (Rows): Row grouping dimensions (*optional*)
- **`c`** (Cols): Column header dimensions (*optional*)
- **`v`** (Values): Value / measure column(s) to aggregate (**required**)
- **`a`** (Aggregation): Aggregation function (`sum`, `mean`, `median`, `min`, `max`, `count`, `std`, `n`; default: `sum`)

### Core Guarantees:
1. **Always Flat 1D Output**: Never produces MultiIndex rows or tuples in column headers. Column names are joined with `_` (e.g. `Sales_2023`).
2. **Always Clean Reset Index**: Returns a standard `RangeIndex(0, 1, 2, ...)` every time.
3. **Automatic Subsetting**: Only references `r`, `c`, and `v`—no need to pre-select columns.
4. **No Silent Data Loss**: `dropna=False` by default, retaining missing categories.

In [ ]:
import pytae as pt
import pandas as pd

penguins = pt.sample("penguins")
flights = pt.sample("flights")
penguins.head()

## 1. Classic 2D Contingency Grid (`r=`, `c=`, `v=`)

Summarize average body mass across islands (rows) and species (columns):

In [ ]:
penguins.pt.pivot(r="island", c="species", v="body_mass_g", a="mean")

## 2. Frequency Matrix via Row Counts (`a="n"`)

Count occurrences across dimensions (the direct successor to cross-tabulation):

In [ ]:
penguins.pt.pivot(r="island", c="species", v="sex", a="n")

## 3. Flexible Excel Layouts: Optional Rows or Columns

Just like dragging fields in Excel, `r` and `c` are completely optional.

In [ ]:
# Omit c: vertical summary table by row group
penguins.pt.pivot(r="species", v="body_mass_g", a="mean")

In [ ]:
# Omit r: horizontal summary table across column headers
penguins.pt.pivot(c="species", v="body_mass_g", a="mean")

In [ ]:
# Omit both r and c: grand scalar aggregation across the dataset
penguins.pt.pivot(v="body_mass_g", a="mean")

## 4. Multi-Level Dimensions & Multi-Metric Reporting

Pass a list of columns to `r`, `c`, or `v`:

In [ ]:
# Hierarchical rows + multiple metric values
penguins.pt.pivot(
    r=["island", "sex"],
    c="species",
    v=["body_mass_g", "flipper_length_mm"],
    a="mean",
    fill_value=0
)

## 5. Seamless Method Chaining & Plotting

Because `pt.pivot` always returns a clean, rectangular DataFrame with a flat RangeIndex, you can immediately chain into further transformations or `.pt.plot()`:

In [ ]:
piv = penguins.pt.pivot(r="island", c="species", v="body_mass_g", a="mean", fill_value=0)
# Clean flat columns and standard index ready for downstream manipulation:
print("Columns:", list(piv.columns))
print("Index:", list(piv.index))
piv